# Luna failure categories

Absolute multi-label counts for the Agentic and Baseline arms. The analysis includes only valid, non-accepted outputs: 87 Agentic and 89 Baseline models. Labels were consolidated using a two-pass Luna consensus with strict arbitration of disagreements. A model may contribute to more than one category, so each row can sum to more than its denominator.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

roots = [Path(os.environ["NEURALCAD_DATA_DIR"])] if "NEURALCAD_DATA_DIR" in os.environ else []
roots += [
    Path("/home/mika/neuralcad-data"),
    Path(r"\\\\wsl.localhost\\Ubuntu-24.04\\home\\mika\\neuralcad-data"),
]
data_root = next(root for root in roots if root.is_dir())

with (data_root / "llm_eval" / "luna_failure_categories.json").open() as file:
    data = json.load(file)


In [ ]:
failure_table = pd.DataFrame.from_dict(
    {arm: values["counts"] for arm, values in data["arms"].items()},
    orient="index",
)
failure_table.index.name = "arm"
failure_table = failure_table.rename(columns=data["category_labels"])
failure_table


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 18,
    "legend.title_fontsize": 18,
})

plot_labels = [
    "Geometric\nquality",
    "Incomplete\nedit",
    "Wrong\nplacement",
    "Incorrect\ndimensions",
    "No effective\nedit",
    "Collateral\ndamage",
    "Incorrect\norientation",
    "Instruction\nmisinterpretation",
]
arm_names = {
    "agentic": "Agentic arm",
    "baseline": "Baseline arm",
}
arm_colors = ["#E83C3C", "#252525"]

plot_table = failure_table.rename(index=arm_names).T
ax = plot_table.plot.bar(
    figsize=(14, 7),
    color=arm_colors,
    width=0.8,
)
ax.set_ylim(0, failure_table.to_numpy().max() * 1.14)
ax.set_xlabel("")
ax.set_ylabel("Number of failed models")
ax.set_xticklabels(plot_labels, rotation=0, ha="center")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="Arm", loc="upper right")
for container in ax.containers:
    ax.bar_label(container, padding=4, fontsize=14)

fig = ax.get_figure()
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.20, top=0.98)
fig.savefig(
    "luna_failure_categories_by_arm.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
)
plt.show()


In [ ]:
horizontal_arm_names = {
    arm: f"{arm_names[arm]} (n={values['denominator']})"
    for arm, values in data["arms"].items()
}
horizontal_plot_table = failure_table.rename(index=horizontal_arm_names).T
horizontal_plot_table = horizontal_plot_table.loc[
    horizontal_plot_table.max(axis=1).sort_values().index
]

ax = horizontal_plot_table.plot.barh(
    figsize=(12, 8),
    color=arm_colors,
    width=0.8,
)
ax.set_xlim(0, failure_table.to_numpy().max() * 1.14)
ax.set_xlabel("Number of failed models")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="Arm", loc="lower right")
for container in ax.containers:
    ax.bar_label(container, padding=4, fontsize=14)

fig = ax.get_figure()
fig.subplots_adjust(left=0.28, right=0.99, bottom=0.12, top=0.98)
fig.savefig(
    "luna_failure_categories_by_arm_horizontal.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
)
plt.show()
